In [ ]:
# ==============================================================================
# PROJETO - CHATBOT BANCÁRIO
# AULA 03 - MACHINE LEARNING / NLU
# ==============================================================================

import pandas as pd
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline


# ==============================================================================
# SCRIPT 1 - GERADOR DE DATASET SINTÉTICO
# ==============================================================================

print("=" * 60)
print("GERANDO DATASET...")
print("=" * 60)

# Templates combinatórios estruturados
templates = {

    'investimentos': {
        's': [
            '',
            'Olá',
            'Bom dia',
            'Por favor',
            'Gostaria de saber'
        ],

        'a': [
            'como aplicar em',
            'quero investir em',
            'qual a rentabilidade do',
            'como funciona o',
            'desejo aplicar no'
        ],

        'o': [
            'tesouro direto',
            'cdb de liquidez diaria',
            'fundo de investimento',
            'lci e lca',
            'mercado de acoes'
        ]
    },

    'consultas': {
        's': [
            '',
            'Oi',
            'Por gentileza',
            'Pode me mostrar',
            'Preciso ver'
        ],

        'a': [
            'quero consultar',
            'onde vejo',
            'qual e o',
            'mostre o',
            'solicito o'
        ],

        'o': [
            'meu saldo atual',
            'extrato da minha conta',
            'comprovante de transferencia',
            'saldo disponivel',
            'historico de transacoes'
        ]
    },

    'pagamentos': {
        's': [
            '',
            'Olá bot',
            'Bom dia',
            'Urgente',
            'Por favor'
        ],

        'a': [
            'quero pagar',
            'como faço para quitar',
            'preciso agendar o pagamento do',
            'como envio um',
            'desejo pagar o'
        ],

        'o': [
            'boleto de luz',
            'codigo de barras',
            'pix para chave email',
            'cartao de credito',
            'imposto veicular'
        ]
    },

    'financiamentos': {
        's': [
            '',
            'Olá',
            'Gostaria de simular',
            'Por gentileza',
            'Preciso de ajuda com'
        ],

        'a': [
            'como contratar',
            'quero simular um',
            'quais as taxas do',
            'como funciona a quitacao do',
            'solicito proposta de'
        ],

        'o': [
            'financiamento imobiliario',
            'credito auto',
            'financiamento de veiculo',
            'credito com garantia',
            'parcelamento da casa propria'
        ]
    }
}


# Lista para armazenar as frases
amostras = []

# Garante que os resultados sejam reproduzíveis
random.seed(42)


# Gerando 25 frases para cada intenção
for intencao, comp in templates.items():

    for _ in range(25):

        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])

        frase = f"{s} {a} {o}".strip().capitalize()

        amostras.append({
            'texto': frase,
            'intencao': intencao
        })


# Criando o DataFrame
df_banco = pd.DataFrame(amostras)


# Salvando o dataset em CSV
df_banco.to_csv(
    'dataset_banco_100.csv',
    index=False,
    encoding='utf-8'
)


print("\nArquivo 'dataset_banco_100.csv' gerado com sucesso!")
print(f"Total de frases: {len(df_banco)}")

print("\nQuantidade de frases por intenção:")
print(df_banco['intencao'].value_counts())


# ==============================================================================
# SCRIPT 2 - PIPELINE DE TREINAMENTO E AVALIAÇÃO
# ==============================================================================

print("\n")
print("=" * 60)
print("TREINANDO O MODELO...")
print("=" * 60)


# Carregando o CSV
df = pd.read_csv('dataset_banco_100.csv')


# Divisão entre treino e teste
X_train, X_test, y_train, y_test = train_test_split(

    df['texto'],
    df['intencao'],

    test_size=0.30,

    random_state=42,

    stratify=df['intencao']
)


# ==============================================================================
# PIPELINE - TF-IDF + KNN
# ==============================================================================

pipeline_nlu = Pipeline([

    (
        'vectorizer',

        TfidfVectorizer(
            ngram_range=(1, 2)
        )
    ),

    (
        'classifier',

        KNeighborsClassifier(
            n_neighbors=3,
            metric='cosine'
        )
    )
])


# Treinando o Pipeline
pipeline_nlu.fit(
    X_train,
    y_train
)


# ==============================================================================
# PREVISÕES
# ==============================================================================

y_pred = pipeline_nlu.predict(
    X_test
)


# ==============================================================================
# MÉTRICAS
# ==============================================================================

acc = accuracy_score(
    y_test,
    y_pred
)


f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)


print("\n")
print("=" * 60)
print("MÉTRICAS GERAIS DO MODELO - KNN")
print("=" * 60)

print(
    f"Acurácia Geral: {acc * 100:.2f}%"
)

print(
    f"F1-Score Geral (Weighted): {f1 * 100:.2f}%"
)

print("=" * 60)


# ==============================================================================
# RELATÓRIO DE CLASSIFICAÇÃO
# ==============================================================================

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ==============================================================================
# GRÁFICO 1 - MATRIZ DE CONFUSÃO
# ==============================================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

labels = sorted(
    df['intencao'].unique()
)


plt.figure(
    figsize=(8, 6)
)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=labels,
    yticklabels=labels
)

plt.title(
    'Matriz de Confusão - KNN (K=3)'
)

plt.xlabel(
    'Classe Preditiva'
)

plt.ylabel(
    'Classe Real'
)

plt.tight_layout()

plt.show()


# ==============================================================================
# GRÁFICO 2 - DISTRIBUIÇÃO DAS CLASSES
# ==============================================================================

plt.figure(
    figsize=(8, 6)
)

sns.countplot(
    data=df,
    x='intencao'
)

plt.title(
    'Distribuição de Frases no Dataset Sintético'
)

plt.xlabel(
    'Intenção'
)

plt.ylabel(
    'Quantidade de Frases'
)

plt.xticks(
    rotation=20
)

plt.tight_layout()

plt.show()


# ==============================================================================
# SCRIPT 3 - CHATBOT INTERATIVO COM FALLBACK
# ==============================================================================

LIMIAR_CONFIANCA = 0.50


def processar_mensagem(texto):

    # Obtendo as probabilidades
    probs = pipeline_nlu.predict_proba(
        [texto]
    )[0]


    # Encontrando a maior probabilidade
    maior_prob = np.max(
        probs
    )


    # Descobrindo a intenção
    intencao = pipeline_nlu.predict(
        [texto]
    )[0]


    # ==================================================================
    # VERIFICAÇÃO DO LIMIAR DE CONFIANÇA
    # ==================================================================

    if maior_prob >= LIMIAR_CONFIANCA:

        print(
            f"\nBot [Intenção: {intencao.upper()} | "
            f"Confiança: {maior_prob * 100:.1f}%]: ",
            end=""
        )


        # --------------------------------------------------------------
        # INVESTIMENTOS
        # --------------------------------------------------------------

        if intencao == "investimentos":

            print(
                "Temos opções de CDB, LCI/LCA e Tesouro Direto. "
                "Qual seu perfil de risco?"
            )


        # --------------------------------------------------------------
        # CONSULTAS
        # --------------------------------------------------------------

        elif intencao == "consultas":

            print(
                "Seu saldo disponível é de R$ 4.820,50. "
                "Deseja consultar seu extrato?"
            )


        # --------------------------------------------------------------
        # PAGAMENTOS
        # --------------------------------------------------------------

        elif intencao == "pagamentos":

            print(
                "Área de pagamentos iniciada. "
                "Por favor, digite ou cole o código de barras."
            )


        # --------------------------------------------------------------
        # FINANCIAMENTOS
        # --------------------------------------------------------------

        elif intencao == "financiamentos":

            print(
                "Simulador de crédito aberto. "
                "Qual o valor do bem que deseja financiar?"
            )


    # ==================================================================
    # FALLBACK
    # ==================================================================

    else:

        print(
            f"\nBot: [FALLBACK - Confiança baixa: "
            f"{maior_prob * 100:.1f}%]"
        )

        print(
            "Desculpe, não consegui entender sua solicitação. "
            "Por favor, aguarde um momento enquanto encaminho "
            "você para um atendente humano..."
        )


# ==============================================================================
# INÍCIO DO CHATBOT
# ==============================================================================

print("\n")
print("=" * 60)
print("=== CHATBOT BANCÁRIO - ATENDIMENTO AO CLIENTE ===")
print("=" * 60)

print(
    "Digite sua mensagem abaixo."
)

print(
    "Para encerrar, digite 'sair'."
)

print("=" * 60)


# Loop de interação
while True:

    entrada = input(
        "\nVocê: "
    ).strip()


    # Verificando encerramento
    if entrada.lower() == 'sair':

        print(
            "\nBot: Atendimento finalizado. "
            "Obrigado por utilizar nossos serviços!"
        )

        break


    # Ignorando mensagens vazias
    if not entrada:

        continue


    # Processando mensagem
    processar_mensagem(
        entrada
    )


    print(
        "-" * 60
    )